# Kakuro Solver — demo

Sube la foto o captura de un Kakuro (3×3 a 7×7, contando las celdas de pistas) y el sistema:

1. encuentra el tablero y corrige la perspectiva,
2. separa celdas blancas y celdas de pista,
3. lee las sumas con Tesseract,
4. resuelve el puzzle con OR-Tools CP-SAT,
5. dibuja la solución sobre la imagen.

Este notebook **no tiene lógica propia**: todo se hace con funciones de `src/kakuro/`.

## 1. Cargar el código

Elige **una** de las dos opciones y escríbela en `OPCION`:

- **`"clonar"`**: cuando el repositorio esté en GitHub, pon su URL en `REPO_URL`.
- **`"subir"`**: mientras no haya GitHub. Comprime la carpeta del proyecto (debe contener `src/` y `requirements.txt`) en un archivo `kakuro-solver.zip` y súbelo cuando se pida.

Fuera de Colab (por ejemplo, en tu computadora con el repositorio ya descargado) esta celda no hace nada y usa la carpeta del repositorio.

In [ ]:
import os
import sys
from pathlib import Path

OPCION = "subir"   # "clonar" o "subir"
REPO_URL = ""      # ejemplo: "https://github.com/usuario/kakuro-solver.git"

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    RAIZ = Path("/content/kakuro-solver")
    if not RAIZ.exists():
        if OPCION == "clonar":
            if not REPO_URL:
                raise ValueError("Escribe la URL del repositorio en REPO_URL.")
            !git clone -q {REPO_URL} {RAIZ}
        else:
            from google.colab import files
            print("Sube kakuro-solver.zip (la carpeta del proyecto comprimida).")
            subido = next(iter(files.upload()))
            !unzip -q -o "{subido}" -d /content/zip_kakuro
            # El zip puede traer la carpeta del proyecto adentro o directamente src/.
            carpeta = next(p.parent for p in Path("/content/zip_kakuro").rglob("requirements.txt"))
            !mv "{carpeta}" {RAIZ}
else:
    # Fuera de Colab: el notebook vive en notebooks/ dentro del repositorio.
    RAIZ = Path.cwd().parent if (Path.cwd().parent / "src").exists() else Path.cwd()

print("Carpeta del proyecto:", RAIZ)

## 2. Instalar dependencias

Tesseract (programa del sistema) y las librerías de `requirements.txt`. En Colab tarda alrededor de un minuto.

In [ ]:
if EN_COLAB:
    !apt-get install -y -qq tesseract-ocr > /dev/null
    %pip install -q -r {RAIZ}/requirements.txt

sys.path.insert(0, str(RAIZ / "src"))

import json
import cv2
import matplotlib.pyplot as plt

from kakuro.estructura import exportar_json
from kakuro.pipeline import leer_imagen, procesar_imagen
from kakuro.visualizacion import dibujar_solucion

!tesseract --version | head -n 1

## 3. Elegir la imagen

En Colab aparece un botón para subir **una** imagen (`.png` o `.jpg`).
Fuera de Colab se usa la ruta de la variable de entorno `KAKURO_IMAGEN`.

In [ ]:
if EN_COLAB:
    from google.colab import files
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError("Selecciona una sola imagen PNG o JPG.")
    RUTA_IMAGEN = next(iter(subidos))
else:
    RUTA_IMAGEN = os.environ["KAKURO_IMAGEN"]

imagen = leer_imagen(RUTA_IMAGEN)
print("Imagen:", Path(RUTA_IMAGEN).name)

## 4. Resolver

`procesar_imagen` corre todo el pipeline y devuelve un `Resultado`. No lanza errores: si algo falla, lo dice en `error` y `error_etapa`.

In [ ]:
resultado = procesar_imagen(imagen)

print("Estado del modelo:", resultado.estado)
if resultado.obtener("error"):
    print(f"Problema en la etapa '{resultado.error_etapa}': {resultado.error}")
if resultado.estado in {"PISTAS_SIN_LECTURA", "PISTAS_FUERA_DE_RANGO"}:
    print("No se leyeron bien algunas pistas: el sistema no inventa sumas y no presenta solución.")
elif resultado.estado == "INFEASIBLE":
    print("Las pistas leídas no permiten una solución con las reglas de Kakuro.")
if resultado.obtener("reintento_bandas"):
    print("Se aplicó el filtro de bandas y se volvieron a leer las pistas.")

## 5. Ver la imagen original y la solución

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 6))
ejes[0].imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
ejes[0].set_title("Imagen original")
if resultado.tiene("imagen"):
    ejes[1].imshow(cv2.cvtColor(dibujar_solucion(resultado.a_dict()), cv2.COLOR_BGR2RGB))
    ejes[1].set_title("Solución" if resultado.solucion is not None else "Sin solución: tablero rectificado")
else:
    ejes[1].set_title("No se encontró el tablero")
for eje in ejes:
    eje.axis("off")
plt.tight_layout()
plt.show()

## 6. Pistas ajustadas y tiempos

Una pista **ajustada** es una suma que el solver cambió respecto de la primera lectura del OCR, porque con la lectura original no había un Kakuro válido (por ejemplo, un 3 leído donde había un 8).

In [ ]:
ajustes = resultado.obtener("ajustes") or []
if ajustes:
    print("Pistas ajustadas (celda, dirección: lectura del OCR -> suma usada):")
    for celda, direccion, candidatos, suma in ajustes:
        print(f"  {celda}, {direccion}: {candidatos[0]} -> {suma}")
elif resultado.solucion is not None:
    print("El solver usó todas las pistas tal como las leyó el OCR.")

print()
print(f"Tiempo de visión (tablero, celdas y OCR): {resultado.tiempo_vision_s:.2f} s")
if resultado.obtener("tiempo_solver_s") is not None:
    print(f"Tiempo del solver (CP-SAT):               {1000 * resultado.tiempo_solver_s:.1f} ms")
print(f"Tiempo total:                             {resultado.tiempo_total_s:.2f} s")

## 7. Estado inicial exportado a JSON

Es lo que el sistema leyó de la imagen antes de resolver, con el mismo formato que `data/ground_truth/`. `pistas_sin_lectura` lista las pistas que el OCR no pudo leer (en `pistas` aparecen como `null`).

In [ ]:
try:
    estado = exportar_json(resultado, "estado_inicial.json", nombre_imagen=Path(RUTA_IMAGEN).name)
    print(json.dumps(estado, ensure_ascii=False))
    print("\nGuardado como estado_inicial.json")
except ValueError as error:
    print("No hay estado inicial para exportar:", error)